# Dynamic Programming and Greedy (Supplement)

This supplemental notebook contains advanced algorithm patterns for this chapter directory.
It is intentionally not listed in the main TOC.

## Dynamic Programming Overview

Dynamic programming is useful when a problem has:
- Optimal substructure
- Overlapping subproblems

Two common styles:
- Top-down memoization
- Bottom-up tabulation

In [1]:
from time import perf_counter
from typing import Dict, List, Optional, Tuple


def fibonacci_naive(n: int) -> int:
    if n <= 1:
        return n
    return fibonacci_naive(n - 1) + fibonacci_naive(n - 2)


def fibonacci_memo(n: int, memo: Optional[Dict[int, int]] = None) -> int:
    if memo is None:
        memo = {}
    if n in memo:
        return memo[n]
    if n <= 1:
        return n
    memo[n] = fibonacci_memo(n - 1, memo) + fibonacci_memo(n - 2, memo)
    return memo[n]


def fibonacci_bottom_up(n: int) -> int:
    if n <= 1:
        return n
    a, b = 0, 1
    for _ in range(2, n + 1):
        a, b = b, a + b
    return b


n = 35
start = perf_counter()
memo_val = fibonacci_memo(n)
memo_t = perf_counter() - start

start = perf_counter()
bottom_val = fibonacci_bottom_up(n)
bottom_t = perf_counter() - start

print(f'fib({n}) memo    = {memo_val} ({memo_t:.6f}s)')
print(f'fib({n}) bottom  = {bottom_val} ({bottom_t:.6f}s)')

fib(35) memo    = 9227465 (0.000026s)
fib(35) bottom  = 9227465 (0.000095s)


## 0/1 Knapsack (DP)

In [2]:
def knapsack(weights: List[int], values: List[int], capacity: int) -> int:
    n = len(weights)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            dp[i][w] = dp[i - 1][w]
            if weights[i - 1] <= w:
                include_value = values[i - 1] + dp[i - 1][w - weights[i - 1]]
                dp[i][w] = max(dp[i][w], include_value)

    return dp[n][capacity]


def knapsack_with_items(weights: List[int], values: List[int], capacity: int) -> Tuple[int, List[int]]:
    n = len(weights)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        for w in range(1, capacity + 1):
            dp[i][w] = dp[i - 1][w]
            if weights[i - 1] <= w:
                include_value = values[i - 1] + dp[i - 1][w - weights[i - 1]]
                dp[i][w] = max(dp[i][w], include_value)

    selected_items = []
    w = capacity
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            selected_items.append(i - 1)
            w -= weights[i - 1]

    return dp[n][capacity], list(reversed(selected_items))


weights = [2, 1, 3, 2, 5]
values = [3, 2, 4, 2, 6]
capacity = 5

max_value, selected = knapsack_with_items(weights, values, capacity)
print('max value:', max_value)
print('selected item indices:', selected)

max value: 7
selected item indices: [0, 2]


## Greedy Overview

Greedy algorithms make locally optimal decisions at each step.
They are fast, but do not guarantee global optimality for every problem.

In [3]:
def activity_selection(start_times: List[int], end_times: List[int]) -> List[int]:
    n = len(start_times)
    if n == 0:
        return []

    activities = [(start_times[i], end_times[i], i) for i in range(n)]
    activities.sort(key=lambda x: x[1])

    selected = [activities[0][2]]
    last_end_time = activities[0][1]

    for start, end, idx in activities[1:]:
        if start >= last_end_time:
            selected.append(idx)
            last_end_time = end

    return selected


def fractional_knapsack(weights: List[int], values: List[int], capacity: int):
    items = [(values[i] / weights[i], weights[i], values[i], i) for i in range(len(weights))]
    items.sort(reverse=True)

    total_value = 0.0
    remaining = capacity
    selected = []

    for ratio, weight, value, idx in items:
        if remaining >= weight:
            total_value += value
            remaining -= weight
            selected.append((idx, 1.0))
        elif remaining > 0:
            frac = remaining / weight
            total_value += value * frac
            selected.append((idx, frac))
            break

    return total_value, selected


start = [1, 3, 0, 5, 8, 5]
end = [2, 4, 6, 7, 9, 9]
print('activity indices:', activity_selection(start, end))

w = [2, 1, 3, 2]
v = [3, 2, 4, 2]
cap = 5
val, picks = fractional_knapsack(w, v, cap)
print('fractional knapsack value:', round(val, 2))
print('fractions:', picks)

activity indices: [0, 1, 3, 4]
fractional knapsack value: 7.67
fractions: [(1, 1.0), (0, 1.0), (2, 0.6666666666666666)]


## When to Prefer DP vs Greedy

- Prefer DP when optimality is required and subproblems overlap.
- Prefer Greedy when the greedy-choice property is proven or when a fast approximation is acceptable.